In [30]:
import pandas as pd

fhv_df = pd.read_csv("fhvhv_tripdata_2025-08.csv")


project the clolumns we need

In [31]:
fhv_df = fhv_df[['hvfhs_license_num', 'request_datetime', 'pickup_datetime','dropoff_datetime', 'PULocationID', 'DOLocationID', 'wav_request_flag', 'wav_match_flag', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee']]

Check if there is any NaN

In [32]:
fhv_df.isna().sum() 

hvfhs_license_num       0
request_datetime        0
pickup_datetime         0
dropoff_datetime        0
PULocationID            0
DOLocationID            0
wav_request_flag        0
wav_match_flag          0
base_passenger_fare     0
tolls                   0
bcf                     0
sales_tax               0
congestion_surcharge    0
airport_fee             0
dtype: int64

convert the request/pickup/dropoff time to datetime

In [33]:

fhv_df["request_datetime"] = pd.to_datetime(fhv_df["request_datetime"], errors="coerce")
fhv_df["pickup_datetime"] = pd.to_datetime(fhv_df["pickup_datetime"], errors="coerce")
fhv_df["dropoff_datetime"] = pd.to_datetime(fhv_df["dropoff_datetime"], errors="coerce")

fhv_df[['request_datetime', 'pickup_datetime', 'dropoff_datetime']].isna().sum()

request_datetime    0
pickup_datetime     0
dropoff_datetime    0
dtype: int64

Check if there's any logically incorrect time: pickup earlier than request, or dropoff earlier than pickup

In [34]:

invalid_time = fhv_df[(fhv_df["pickup_datetime"] <= fhv_df["request_datetime"]) | (fhv_df["dropoff_datetime"] <= fhv_df["pickup_datetime"])]
len(invalid_time)  

224540

Filter out those logically incorrect

In [35]:
fhv_df = fhv_df[(fhv_df["pickup_datetime"] > fhv_df["request_datetime"]) & (fhv_df["dropoff_datetime"] > fhv_df["pickup_datetime"])]

Augment a new column temporarily for analysis

In [36]:
fhv_df["trip_minutes"] = (
    (fhv_df["dropoff_datetime"] - fhv_df["pickup_datetime"]).dt.total_seconds() / 60
)



Only keep those data with a reasonable trip minutes (ride between 5 and 180 min)

In [37]:
fhv_df = fhv_df[(fhv_df["trip_minutes"] >= 5) & (fhv_df["trip_minutes"] < 180)]

Only keep trip data with a positive fee

In [40]:
fhv_df = fhv_df[(fhv_df["base_passenger_fare"] > 0) & (fhv_df["tolls"] >= 0) & (fhv_df["bcf"] >= 0) & (fhv_df["sales_tax"] >= 0) & (fhv_df["congestion_surcharge"] >= 0) & (fhv_df["airport_fee"] >= 0)]

Check if any data with location ID not in NYC or unknown (i.e. location id outside [1,263])

In [41]:
invalid_loc = fhv_df[
    (fhv_df["PULocationID"] <= 0) |
    (fhv_df["DOLocationID"] <= 0) |
    (fhv_df["PULocationID"] > 263) |
    (fhv_df["DOLocationID"] > 263)
]
len(invalid_loc), invalid_loc.head()

(890285,
    hvfhs_license_num    request_datetime     pickup_datetime  \
 24            HV0003 2025-08-01 00:05:07 2025-08-01 00:13:19   
 30            HV0005 2025-07-31 23:58:03 2025-08-01 00:04:06   
 67            HV0003 2025-08-01 00:34:13 2025-08-01 00:38:56   
 79            HV0003 2025-08-01 00:13:55 2025-08-01 00:19:44   
 88            HV0003 2025-08-01 00:48:06 2025-08-01 00:50:54   
 
       dropoff_datetime  PULocationID  DOLocationID wav_request_flag  \
 24 2025-08-01 00:33:46           186           265                N   
 30 2025-08-01 00:34:08           186           265                N   
 67 2025-08-01 01:07:35           138           265                N   
 79 2025-08-01 01:05:33           163           265                N   
 88 2025-08-01 01:13:34           137           265                N   
 
    wav_match_flag  base_passenger_fare  tolls   bcf  sales_tax  \
 24              N                64.91  16.06  1.59       0.00   
 30              N             

We only keep data with location ID in NYC and not unknown (i.e. 1 <= id <= 263)

In [42]:
fhv_df = fhv_df[(fhv_df['PULocationID']<=263) & (fhv_df['PULocationID']>=1)]
fhv_df = fhv_df[(fhv_df['DOLocationID']<=263) & (fhv_df['DOLocationID']>=1)]

In [43]:
fhv_df.columns

Index(['hvfhs_license_num', 'request_datetime', 'pickup_datetime',
       'dropoff_datetime', 'PULocationID', 'DOLocationID', 'wav_request_flag',
       'wav_match_flag', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax',
       'congestion_surcharge', 'airport_fee', 'trip_minutes'],
      dtype='object')

Calculate total price for every ride

In [44]:
fhv_df['total_amount'] = fhv_df['base_passenger_fare'] + fhv_df['tolls'] + fhv_df['bcf'] + fhv_df['sales_tax'] + fhv_df['congestion_surcharge'] + fhv_df['airport_fee']

rename columns to match those in our schema

In [45]:
fhv_df = fhv_df.rename(columns={'hvfhs_license_num': 'service_provider', 'PULocationID': 'pickup_location', 'DOLocationID': 'dropoff_location'})

Drop unnecessary columns

In [46]:
fhv_df = fhv_df.drop(columns=["trip_minutes","base_passenger_fare", "tolls", "bcf", "sales_tax", "congestion_surcharge", "airport_fee"])

In [47]:
fhv_df.to_csv('fhv_cleaned_df.csv')